# EX4 — Subgroup Fairness & Consistency Analysis

**Research Question 3**: How consistent is model performance across patient subgroups, and does conditioning reduce performance disparities between groups?

**Models compared**:
- `std_trial_036` — Best standard U-Net from EX1 (Baseline Champion)
- `feat_trial_015` — Best FiLM model using 22 selected features from EX3 (FiLM Champion)

**Analysis steps**:
1. Compute Brain Parenchymal Fraction (BPF) for GT stratification — cached after first run
2. Define 6 clinically-motivated subgroup splits
3. Per-subgroup SSIM, L1, Dice, VS metrics table
4. BPF prediction accuracy per atrophy group
5. Feature input sensitivity visualization

In [ ]:
import json
import sys
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats
import nibabel as nib
from tqdm.auto import tqdm

sys.path.insert(0, '.')

# ── Paths ─────────────────────────────────────────────────────────────────
STD_DIR          = Path('out/std_trial_036')
FILM_DIR         = Path('out/ex3/bayes_search/feat_trial_015')
META_PATH        = Path('data/metadata/metadata_combined.csv')
SPLITS_PATH      = Path('data/metadata/splits.json')
HUNT3_DIR        = Path('data/HUNT3')
HUNT4_DIR        = Path('data/HUNT4')
HUNT4_GT_SEG_DIR = Path('out/hunt4_gt/segmentations')
BPF_CACHE_PATH   = Path('out/ex4/bpf_cache.csv')
FIG_DIR          = Path('out/overleaf_figures/ex4')

FIG_DIR.mkdir(parents=True, exist_ok=True)
Path('out/ex4').mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────
STD_COLOR  = '#4C72B0'
FILM_COLOR = '#17A2B8'
N_MIN      = 10
OUTLIERS   = set()  # subject IDs withheld for privacy; populate to flag specific outliers
CROP_AXES  = ((16, 10, 0), (17, 11, 17))

# FastSurfer label sets (from model_analysis/evaluate_models.py)
GM_LABELS  = frozenset([*range(1000,1036), *range(2000,2036),
                         10,11,12,13,17,18,26,28,49,50,51,52,53,54,58,60])
WM_LABELS  = frozenset([2,41,7,46,77,85,251,252,253,254,255])
CSF_LABELS = frozenset([4,5,14,15,24,31,43,44,63,72])

with open(SPLITS_PATH) as f:
    splits = json.load(f)
test_ids = [str(x).zfill(5) for x in splits['test']]
print(f'Test subjects: {len(test_ids)}')
print(f'Sample IDs: {test_ids[:5]}')

---
## Step 1 — Brain Parenchymal Fraction (BPF) Computation

**BPF formula**: BPF = (GM_volume + WM_volume) / ICV, where ICV = GM + WM + CSF

**Sources used**:
- `bpf_h3_seg3` — HUNT3 baseline from SEG_3 (label 1=CSF, 2=GM, 3=WM — confirmed by T1 intensity analysis)
- `bpf_h4_seg3` — HUNT4 GT from SEG_3 (consistent ΔBPF for stratification)
- `bpf_h4_gt_fs` — HUNT4 GT from FastSurfer `gt_seg.nii.gz`
- `bpf_h4_std_fs` / `bpf_h4_film_fs` — Predicted HUNT4 from FastSurfer `pred_seg.nii.gz`

**ΔBPF for atrophy stratification**: uses SEG_3 both timepoints (internally consistent).  
**ΔBPF for model comparison**: SEG_3 H3 vs FastSurfer H4 (systematic scale offset; rank ordering valid).

> Results cached to `out/ex4/bpf_cache.csv` — second run loads instantly.

In [ ]:
def compute_bpf_seg3(subject_id: str, timepoint: int) -> dict:
    """BPF from simplified 3-class SEG_3 segmentation. label1=CSF, 2=GM, 3=WM"""
    hunt_dir = HUNT3_DIR if timepoint == 0 else HUNT4_DIR
    path = hunt_dir / subject_id / f'{subject_id}_{timepoint}_SEG_3_PREP_MNI.nii.gz'
    seg  = nib.load(str(path)).get_fdata()
    gm   = int((seg == 2).sum())
    wm   = int((seg == 3).sum())
    csf  = int((seg == 1).sum())
    icv  = gm + wm + csf
    return {'bpf': (gm + wm) / icv if icv > 0 else float('nan'),
            'gm': gm, 'wm': wm, 'csf': csf, 'icv': icv}


def compute_bpf_fastsurfer(seg_path) -> dict:
    """BPF from FastSurfer full-atlas segmentation (FreeSurfer label space)."""
    seg = nib.load(str(seg_path)).get_fdata()
    gm  = int(np.isin(seg, list(GM_LABELS)).sum())
    wm  = int(np.isin(seg, list(WM_LABELS)).sum())
    csf = int(np.isin(seg, list(CSF_LABELS)).sum())
    icv = gm + wm + csf
    return {'bpf': (gm + wm) / icv if icv > 0 else float('nan'),
            'gm': gm, 'wm': wm, 'csf': csf, 'icv': icv}


print('BPF helpers defined.')

In [ ]:
if BPF_CACHE_PATH.exists():
    print(f'Loading BPF cache from {BPF_CACHE_PATH} ...')
    bpf_df = pd.read_csv(BPF_CACHE_PATH, dtype={'hunt_id': str})
    bpf_df['hunt_id'] = bpf_df['hunt_id'].str.zfill(5)
    print(f'  Loaded {len(bpf_df)} subjects.')
else:
    print('Computing BPF for 113 test subjects (~10 min) ...')
    rows, missing = [], []
    for sid in tqdm(test_ids, desc='BPF'):
        row = {'hunt_id': sid}
        for key, fn in [
            ('h3_seg3',  lambda s: compute_bpf_seg3(s, 0)),
            ('h4_seg3',  lambda s: compute_bpf_seg3(s, 1)),
            ('h4_gt_fs', lambda s: compute_bpf_fastsurfer(HUNT4_GT_SEG_DIR / s / 'gt_seg.nii.gz')),
            ('h4_std_fs',  lambda s: compute_bpf_fastsurfer(STD_DIR  / 'segmentations' / s / 'pred_seg.nii.gz')),
            ('h4_film_fs', lambda s: compute_bpf_fastsurfer(FILM_DIR / 'segmentations' / s / 'pred_seg.nii.gz')),
        ]:
            try:
                d = fn(sid)
                for metric in ['bpf','gm','wm','csf','icv']:
                    row[f'{metric}_{key}'] = d[metric]
            except Exception as e:
                missing.append(f'{sid}/{key}: {e}')
                for metric in ['bpf','gm','wm','csf','icv']:
                    row[f'{metric}_{key}'] = float('nan')
        rows.append(row)

    bpf_df = pd.DataFrame(rows)
    bpf_df.to_csv(BPF_CACHE_PATH, index=False)
    print(f'Saved cache -> {BPF_CACHE_PATH}')
    if missing:
        print(f'Missing ({len(missing)}): {missing[:5]}')

print(f'\nColumns: {list(bpf_df.columns)}')
bpf_df.head(3)

In [ ]:
# Derive ΔBPF columns
bpf_df['dbpf_gt_seg3']  = bpf_df['bpf_h4_seg3']    - bpf_df['bpf_h3_seg3']
bpf_df['dbpf_gt_fs']    = bpf_df['bpf_h4_gt_fs']   - bpf_df['bpf_h3_seg3']
bpf_df['dbpf_std_fs']   = bpf_df['bpf_h4_std_fs']  - bpf_df['bpf_h3_seg3']
bpf_df['dbpf_film_fs']  = bpf_df['bpf_h4_film_fs'] - bpf_df['bpf_h3_seg3']

summary = [
    ('GT ΔBPF (SEG3 both timepoints)',   'dbpf_gt_seg3'),
    ('GT ΔBPF (FS HUNT4 − SEG3 HUNT3)', 'dbpf_gt_fs'),
    ('Baseline ΔBPF (FS pred − SEG3)',   'dbpf_std_fs'),
    ('FiLM ΔBPF (FS pred − SEG3)',       'dbpf_film_fs'),
]
print(f'BPF Summary {"-"*50}')
print(f'{"Source":<40}  {"mean":>8}  {"std":>8}  {"min":>8}  {"max":>8}')
print('-' * 80)
for name, col in summary:
    s = bpf_df[col].dropna()
    print(f'{name:<40}  {s.mean():8.5f}  {s.std():8.5f}  {s.min():8.5f}  {s.max():8.5f}')
print()
print('Expected: negative ΔBPF = brain parenchyma loss over time')
print('Note: SEG3 and FastSurfer use different label spaces; absolute values differ.')

In [ ]:
# ── Step 1 Visualizations: BPF at HUNT3 vs HUNT4 ───────────────────────────
H3_COLOR = '#4878CF'   # blue  – HUNT3 baseline
H4_COLOR = '#D65F5F'   # red   – HUNT4 follow-up

valid_bpf = bpf_df.dropna(subset=['bpf_h3_seg3', 'bpf_h4_seg3']).copy()
delta      = valid_bpf['dbpf_gt_seg3']

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# ── 1a: Box plots with jitter ─────────────────────────────────────────────
ax = axes[0]
bp = ax.boxplot(
    [valid_bpf['bpf_h3_seg3'].values, valid_bpf['bpf_h4_seg3'].values],
    labels=['HUNT3', 'HUNT4'],
    patch_artist=True, widths=0.45,
    flierprops=dict(marker='o', markersize=3, alpha=0.4),
    medianprops=dict(color='black', linewidth=2),
    whiskerprops=dict(color='gray'), capprops=dict(color='gray'),
)
bp['boxes'][0].set_facecolor(H3_COLOR); bp['boxes'][0].set_alpha(0.55)
bp['boxes'][1].set_facecolor(H4_COLOR); bp['boxes'][1].set_alpha(0.55)

np.random.seed(42)
for xi, col, color in [(1, 'bpf_h3_seg3', H3_COLOR), (2, 'bpf_h4_seg3', H4_COLOR)]:
    vals   = valid_bpf[col].values
    jitter = np.random.uniform(-0.10, 0.10, len(vals))
    ax.scatter(xi + jitter, vals, color=color, alpha=0.35, s=14, zorder=3)

ax.set_ylabel('BPF', fontsize=11)
ax.set_title('BPF Distribution: HUNT3 vs HUNT4', fontsize=10, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')

for xi, col in [(1, 'bpf_h3_seg3'), (2, 'bpf_h4_seg3')]:
    m = valid_bpf[col].mean()
    ax.annotate(f'mean = {m:.4f}', xy=(xi, m), xytext=(xi + 0.28, m),
                fontsize=8, color='black', va='center',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

ax.legend(
    handles=[mpatches.Patch(facecolor=H3_COLOR, alpha=0.7, label=f'HUNT3  (n={len(valid_bpf)})'),
             mpatches.Patch(facecolor=H4_COLOR, alpha=0.7, label=f'HUNT4  (n={len(valid_bpf)})')],
    fontsize=9, loc='lower right')

# ── 1b: Scatter — HUNT3 BPF vs HUNT4 BPF per subject ────────────────────
ax = axes[1]

sc = ax.scatter(
    valid_bpf['bpf_h3_seg3'], valid_bpf['bpf_h4_seg3'],
    c=delta, cmap='RdYlGn',
    vmin=delta.quantile(0.05), vmax=delta.quantile(0.95),
    s=45, alpha=0.85, edgecolors='gray', linewidths=0.3, zorder=3,
)

lo = min(valid_bpf['bpf_h3_seg3'].min(), valid_bpf['bpf_h4_seg3'].min()) - 0.005
hi = max(valid_bpf['bpf_h3_seg3'].max(), valid_bpf['bpf_h4_seg3'].max()) + 0.005
ax.plot([lo, hi], [lo, hi], 'k--', lw=1.5, alpha=0.5, label='Identity (no change)')
ax.fill_between([lo, hi], [lo, lo], [lo, hi], alpha=0.04, color='red',
                label='Below line = atrophy (HUNT4 < HUNT3)')

ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_xlabel('BPF HUNT3', fontsize=11)
ax.set_ylabel('BPF HUNT4', fontsize=11)
ax.set_title('BPF per Subject: HUNT3 to HUNT4', fontsize=10, fontweight='bold')
ax.grid(alpha=0.3, linestyle='--')
ax.legend(fontsize=8, loc='upper left')

cbar = fig.colorbar(sc, ax=ax, shrink=0.80, pad=0.02)
cbar.set_label('\u0394BPF = HUNT4 - HUNT3', fontsize=9)

fig.suptitle(
    f'Brain Parenchymal Fraction - Test Set  (n = {len(valid_bpf)} subjects)',
    fontsize=10, y=1.03,
)
plt.tight_layout()
plt.savefig(FIG_DIR / 'bpf_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {FIG_DIR}/bpf_distributions.png')
print(f'\nSummary:')
print(f'  Mean BPF HUNT3 : {valid_bpf["bpf_h3_seg3"].mean():.5f}  \u00b1 {valid_bpf["bpf_h3_seg3"].std():.5f}')
print(f'  Mean BPF HUNT4 : {valid_bpf["bpf_h4_seg3"].mean():.5f}  \u00b1 {valid_bpf["bpf_h4_seg3"].std():.5f}')
print(f'  Mean \u0394BPF       : {delta.mean():.5f}  \u00b1 {delta.std():.5f}')
print(f'  Subjects with atrophy (\u0394BPF < 0): {(delta < 0).sum()} / {len(delta)}')


In [ ]:
# Subjects with positive ΔBPF (BPF increased HUNT3 → HUNT4)
gainers = valid_bpf[valid_bpf['dbpf_gt_seg3'] > 0][['hunt_id', 'bpf_h3_seg3', 'bpf_h4_seg3', 'dbpf_gt_seg3']]
gainers = gainers.sort_values('dbpf_gt_seg3', ascending=False)

print(f'Subjects with positive ΔBPF: {len(gainers)}')
print()
print(gainers.to_string(index=False))


---
## Step 2 — Subgroup Feature Setup

Six features selected based on EX2 SHAP ranking + clinical interest:

| Feature | Type | Strategy | SHAP Rank |
|---------|------|----------|-----------|
| Sex | Binary | 0=Female / 1=Male | #1 |
| Birth Year / Age | Continuous | Median split (young-old vs old-old) | #2 |
| Waist Circumference | Continuous | Median split (low vs high) | #3 |
| Smoking Status | Categorical | 4 categories (Never/Former/Current-light/Current-heavy) | #4 |
| Education | Categorical | 6 categories | #5 |
| Diabetes | Binary | 0=No / 1=Yes | #26 (added for clinical interest) |

All continuous features are normalized [0,1]. **BirthYear**: 0=oldest, 1=youngest.  
Median splits use the **test-set median** (n=113 subjects).

In [ ]:
meta = pd.read_csv(META_PATH, dtype={'hunt_id': str})
meta['hunt_id'] = meta['hunt_id'].str.zfill(5)

def recover_categorical(df: pd.DataFrame, prefix: str) -> pd.Series:
    cols = sorted([c for c in df.columns if c.startswith(prefix + '_')])
    return df[cols].idxmax(axis=1).str.replace(f'{prefix}_', '', regex=False).astype(int)

meta['SmoStat']   = recover_categorical(meta, 'SmoStat@NT3BLQ1')
meta['Education'] = recover_categorical(meta, 'Educ@NT2BLQ1')
meta_test = meta[meta['hunt_id'].isin(test_ids)].copy().reset_index(drop=True)
print(f'Test subjects in metadata: {len(meta_test)}')

In [ ]:
SMOKING_LABELS   = {0:'Never', 1:'Former', 2:'Daily', 3:'Sometimes'}
EDUCATION_LABELS = {0:'Primary', 1:'High School', 2:'High School',
                    3:'Uni Qualifying', 4:'University <4', 5:'University 4+'}
SEX_LABELS       = {0:'Female', 1:'Male'}
DIABETES_LABELS  = {0:'No', 1:'Yes'}

age_med   = meta_test['BirthYear'].median()
waist_med = meta_test['WaistCirc@NT3BLM'].median()
print(f'BirthYear test-set median: {age_med:.4f}')
print(f'  <= {age_med:.4f} -> Oldest half  |  > {age_med:.4f} -> Youngest half')
print(f'  (0.0=oldest cohort, 1.0=youngest; median {age_med:.2f} means roughly equal split)')
print(f'WaistCirc test-set median: {waist_med:.4f}')

meta_test = meta_test.copy()
meta_test['grp_Sex']      = meta_test['Sex'].map(SEX_LABELS)
meta_test['grp_Age']      = meta_test['BirthYear'].apply(
    lambda v: 'Oldest half' if v <= age_med else 'Youngest half')
meta_test['grp_Waist']    = meta_test['WaistCirc@NT3BLM'].apply(
    lambda v: 'Low' if v <= waist_med else 'High')
meta_test['grp_Smoking']  = meta_test['SmoStat'].map(SMOKING_LABELS)
meta_test['grp_Education']= meta_test['Education'].map(EDUCATION_LABELS)
meta_test['grp_Diabetes'] = meta_test['DiaEv@NT3BLQ1'].map(DIABETES_LABELS)

SUBGROUP_CONFIG = [
    ('Sex',       'grp_Sex',        ['Female', 'Male']),
    ('Age',       'grp_Age',        ['Oldest half', 'Youngest half']),
    ('WaistCirc', 'grp_Waist',      ['Low', 'High']),
    ('Smoking',   'grp_Smoking',    ['Never', 'Former', 'Daily', 'Sometimes']),
    ('Education', 'grp_Education',  ['Primary', 'High School', 'Uni Qualifying', 'University <4', 'University 4+']),
    ('Diabetes',  'grp_Diabetes',   ['No', 'Yes']),
]

print('\nSubgroup sizes (test set, n=113):')
print(f'{"Feature":<12} {"Group":<20} {"n":>5}  flag')
print('-'*52)
for feat, col, order in SUBGROUP_CONFIG:
    for grp in order:
        n = int((meta_test[col] == grp).sum())
        if n == 0: continue
        flag = ' << small n' if n < N_MIN else ''
        print(f'{feat:<12} {grp:<20} {n:>5}{flag}')
    print()

---
## Step 3 — Per-Subgroup Metrics Table

Metrics: SSIM loss, L1 loss, Dice GM/WM/CSF, VS GM/WM/CSF.  
All values are directly read from the pre-computed `dice_results.csv` files.

**Disparity metric**: within-model gap = max_group − min_group for a given feature.  
**Δ-of-Δ** = FiLM gap − Baseline gap (negative = FiLM reduces disparity; more equitable).

In [ ]:
def load_metrics(csv_path: Path, model_name: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df['ssim_l1_loss'] = 0.8 * df['ssim_loss'] + 0.2 * df['l1_loss']
    df['model']   = model_name
    df['hunt_id'] = df['patient_id'].astype(str).str.zfill(5)
    return df

std_metrics  = load_metrics(STD_DIR  / 'dice_results.csv', 'Baseline')
film_metrics = load_metrics(FILM_DIR / 'dice_results.csv', 'FiLM')
all_metrics  = pd.concat([std_metrics, film_metrics], ignore_index=True)

METRIC_COLS = ['ssim_loss','l1_loss','ssim_l1_loss',
               'dice_GM','dice_WM','dice_CSF',
               'vs_GM','vs_WM','vs_CSF']

print(f'Baseline: {len(std_metrics)} subjects')
print(f'FiLM:     {len(film_metrics)} subjects')
print(f'Metrics: {METRIC_COLS}')

In [ ]:
grp_cols  = ['hunt_id'] + [col for _, col, _ in SUBGROUP_CONFIG]
meta_grps = meta_test[grp_cols].copy()
merged = all_metrics.merge(meta_grps, on='hunt_id', how='inner')
print(f'Merged rows: {len(merged)} ({len(merged)//2} subjects x 2 models)')

In [ ]:
def build_subgroup_table(merged_df, subgroup_config, metric_cols):
    rows = []
    for feat_name, grp_col, order in subgroup_config:
        for grp in order:
            mask_grp = merged_df[grp_col] == grp
            n = int(mask_grp.sum()) // 2
            if n == 0: continue
            for model in ['Baseline', 'FiLM']:
                subset = merged_df[mask_grp & (merged_df['model'] == model)]
                row = {'feature': feat_name, 'group': grp, 'model': model,
                       'n': n, 'small_n': n < N_MIN}
                for m in metric_cols:
                    row[f'mean_{m}'] = float(subset[m].mean())
                    row[f'std_{m}']  = float(subset[m].std())
                rows.append(row)
    return pd.DataFrame(rows)

subgroup_tbl = build_subgroup_table(merged, SUBGROUP_CONFIG, METRIC_COLS)
print(f'Subgroup table: {len(subgroup_tbl)} rows')
subgroup_tbl.head(6)

In [ ]:
print('=== Combined Loss (0.8·SSIM + 0.2·L1) per Subgroup (lower = better) ===')
print(f'{"Feature":<12} {"Group":<22} {"n":>4}  {"Baseline":>10}  {"FiLM":>10}  {"Delta":>12}  dir')
print('-'*78)
for feat_name, grp_col, order in SUBGROUP_CONFIG:
    for grp in order:
        sub = subgroup_tbl[(subgroup_tbl['feature']==feat_name) & (subgroup_tbl['group']==grp)]
        if len(sub) < 2: continue
        base_r = sub[sub['model']=='Baseline'].iloc[0]
        film_r = sub[sub['model']=='FiLM'].iloc[0]
        n   = base_r['n']; flg = '<<' if n < N_MIN else '  '
        b   = base_r['mean_ssim_l1_loss']
        f   = film_r['mean_ssim_l1_loss']
        d   = f - b
        dir_sym = 'better' if d < 0 else 'worse '
        print(f'{feat_name:<12} {grp:<22} {n:>4}{flg}  {b:10.4f}  {f:10.4f}  {d:+12.4f}  {dir_sym}')


In [ ]:
print('=== Mean Dice (GM+WM+CSF)/3 per Subgroup (higher = better) ===')
print('Macro-average Dice across all three tissue classes.')
print(f'{"Feature":<12} {"Group":<22} {"n":>4}  {"Baseline":>10}  {"FiLM":>10}  {"Delta":>12}  dir')
print('-'*78)
for feat_name, grp_col, order in SUBGROUP_CONFIG:
    for grp in order:
        sub = subgroup_tbl[(subgroup_tbl['feature']==feat_name) & (subgroup_tbl['group']==grp)]
        if len(sub) < 2: continue
        base_r = sub[sub['model']=='Baseline'].iloc[0]
        film_r = sub[sub['model']=='FiLM'].iloc[0]
        n   = base_r['n']; flg = '<<' if n < N_MIN else '  '
        b   = (base_r['mean_dice_GM'] + base_r['mean_dice_WM'] + base_r['mean_dice_CSF']) / 3
        f   = (film_r['mean_dice_GM'] + film_r['mean_dice_WM'] + film_r['mean_dice_CSF']) / 3
        d   = f - b
        dir_sym = 'better' if d > 0 else 'worse '
        print(f'{feat_name:<12} {grp:<22} {n:>4}{flg}  {b:10.4f}  {f:10.4f}  {d:+12.4f}  {dir_sym}')


In [ ]:
# Full delta table — combined loss + per-class Dice  (all: FiLM − Baseline)
# Comb: lower=better | Dice: higher=better
hdr = f'{"Feature":<10} {"Group":<20} {"n":>4}|  {"Comb":>7}  {"dGM":>7}  {"dWM":>7}  {"dCSF":>7}'
print(hdr)
print('-'*(len(hdr)+5))
for feat_name, grp_col, order in SUBGROUP_CONFIG:
    for grp in order:
        sub = subgroup_tbl[(subgroup_tbl['feature']==feat_name) & (subgroup_tbl['group']==grp)]
        if len(sub) < 2: continue
        base_r = sub[sub['model']=='Baseline'].iloc[0]
        film_r = sub[sub['model']=='FiLM'].iloc[0]
        n   = base_r['n']; flg = '*' if n < N_MIN else ' '

        d_comb = film_r['mean_ssim_l1_loss'] - base_r['mean_ssim_l1_loss']
        d_gm   = film_r['mean_dice_GM']  - base_r['mean_dice_GM']
        d_wm   = film_r['mean_dice_WM']  - base_r['mean_dice_WM']
        d_csf  = film_r['mean_dice_CSF'] - base_r['mean_dice_CSF']

        print(f'{feat_name:<10} {grp:<20} {n:>4}{flg}|  {d_comb:+7.4f}  {d_gm:+7.4f}  {d_wm:+7.4f}  {d_csf:+7.4f}')


In [ ]:
disp_rows = []
for feat_name, grp_col, order in SUBGROUP_CONFIG:
    for model in ['Baseline','FiLM']:
        sub = subgroup_tbl[(subgroup_tbl['feature']==feat_name) & (subgroup_tbl['model']==model)]
        sub_valid = sub[~sub['small_n']] if len(sub[~sub['small_n']]) >= 2 else sub
        for metric in METRIC_COLS:
            vals = sub_valid[f'mean_{metric}'].dropna()
            if len(vals) >= 2:
                gap = float(vals.max() - vals.min())
                disp_rows.append({'feature': feat_name, 'model': model,
                                  'metric': metric, 'gap': gap})

disp_df  = pd.DataFrame(disp_rows)
disp_wide = disp_df.pivot_table(
    index=['feature','metric'], columns='model', values='gap').reset_index()
disp_wide.columns.name = None
if 'Baseline' in disp_wide.columns and 'FiLM' in disp_wide.columns:
    disp_wide['delta_of_delta'] = disp_wide['FiLM'] - disp_wide['Baseline']

# Show only SSIM and Dice GM for clarity
for mfilter in ['ssim_loss', 'dice_GM']:
    sub = disp_wide[disp_wide['metric'] == mfilter]
    print(f'\nDisparity ({mfilter}): max_group - min_group')
    print(f'{"Feature":<12} {"Baseline":>12} {"FiLM":>10} {"D-of-D":>10}  note')
    print('-'*62)
    for _, r in sub.iterrows():
        dod  = r.get('delta_of_delta', float('nan'))
        note = '(FiLM fairer)' if not pd.isna(dod) and dod < -0.0001 else \
               '(FiLM wider)' if not pd.isna(dod) and dod > 0.0001 else '(same)'
        print(f'{r["feature"]:<12} {r["Baseline"]:>12.4f} {r["FiLM"]:>10.4f} {dod:>+10.4f}  {note}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

legend_handles = [
    mpatches.Patch(color=STD_COLOR,  label='Baseline (std_trial_036)'),
    mpatches.Patch(color=FILM_COLOR, label='FiLM (feat_trial_015)'),
]

for idx, (feat_name, grp_col, order) in enumerate(SUBGROUP_CONFIG):
    ax = axes[idx]
    plot_data = []
    for grp in order:
        mask = merged[grp_col] == grp
        for model, color in [('Baseline', STD_COLOR), ('FiLM', FILM_COLOR)]:
            for v in merged[mask & (merged['model'] == model)]['ssim_loss'].values:
                plot_data.append({'Group': grp, 'Model': model, 'SSIM Loss': v})
    plot_df = pd.DataFrame(plot_data)
    present = [g for g in order if (plot_df['Group'] == g).any()]
    if not plot_df.empty:
        sns.boxplot(data=plot_df, x='Group', y='SSIM Loss', hue='Model',
                    order=present, palette={'Baseline': STD_COLOR, 'FiLM': FILM_COLOR},
                    ax=ax, width=0.6, fliersize=3, linewidth=1.2)
    ax.set_title(feat_name, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('SSIM Loss' if idx % 3 == 0 else '')
    ax.tick_params(axis='x', labelsize=8, rotation=20)
    if ax.get_legend() is not None: ax.get_legend().remove()
    for i, grp in enumerate(present):
        n = int((meta_test[grp_col] == grp).sum())
        ax.text(i, ax.get_ylim()[0], f'n={n}' + ('⚠' if n<N_MIN else ''),
                ha='center', va='bottom', fontsize=7, color='gray')

fig.legend(handles=legend_handles, loc='upper right', fontsize=10)
fig.suptitle('SSIM Loss by Subgroup — Baseline vs FiLM (lower = better)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'subgroup_ssim_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {FIG_DIR}/subgroup_ssim_boxplots.png')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for idx, (feat_name, grp_col, order) in enumerate(SUBGROUP_CONFIG):
    ax = axes[idx]
    plot_data = []
    for grp in order:
        mask = merged[grp_col] == grp
        for model in ['Baseline','FiLM']:
            for v in merged[mask & (merged['model']==model)]['dice_GM'].values:
                plot_data.append({'Group': grp, 'Model': model, 'Dice GM': v})
    plot_df = pd.DataFrame(plot_data)
    present = [g for g in order if (plot_df['Group']==g).any()]
    if not plot_df.empty:
        sns.boxplot(data=plot_df, x='Group', y='Dice GM', hue='Model',
                    order=present, palette={'Baseline':STD_COLOR,'FiLM':FILM_COLOR},
                    ax=ax, width=0.6, fliersize=3, linewidth=1.2)
    ax.set_title(feat_name, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Dice GM' if idx%3==0 else '')
    ax.tick_params(axis='x', labelsize=8, rotation=20)
    if ax.get_legend() is not None: ax.get_legend().remove()
    for i, grp in enumerate(present):
        n = int((meta_test[grp_col]==grp).sum())
        ax.text(i, ax.get_ylim()[0], f'n={n}'+('⚠' if n<N_MIN else ''),
                ha='center', va='bottom', fontsize=7, color='gray')

fig.legend(handles=legend_handles, loc='upper right', fontsize=10)
fig.suptitle('Dice GM by Subgroup — Baseline vs FiLM (higher = better)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'subgroup_dice_gm_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {FIG_DIR}/subgroup_dice_gm_boxplots.png')

In [ ]:
disp_ssim = disp_wide[disp_wide['metric']=='ssim_loss'].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(10,5))
x = np.arange(len(disp_ssim)); w = 0.35
ax.bar(x - w/2, disp_ssim['Baseline'], w, color=STD_COLOR,  label='Baseline', alpha=0.9)
ax.bar(x + w/2, disp_ssim['FiLM'],     w, color=FILM_COLOR, label='FiLM',     alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(disp_ssim['feature'], fontsize=11)
ax.set_ylabel('SSIM Loss — within-feature group gap'); ax.set_title(
    'Performance Disparity per Feature (gap = max_group − min_group; lower = fairer)')
ax.legend(); ax.spines[['top','right']].set_visible(False)
for i, (_, r) in enumerate(disp_ssim.iterrows()):
    dod = r.get('delta_of_delta', float('nan'))
    color = 'green' if not pd.isna(dod) and dod < 0 else 'red'
    ax.text(i, max(r['Baseline'], r['FiLM'])+0.0003, f'D={dod:+.4f}',
            ha='center', va='bottom', fontsize=8, color=color)
plt.tight_layout()
plt.savefig(FIG_DIR / 'disparity_bar_ssim.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {FIG_DIR}/disparity_bar_ssim.png')

In [ ]:
print('=== Outlier Subjects ===')
outlier_meta = meta_test[meta_test['hunt_id'].isin(OUTLIERS)]
for _, row in outlier_meta.iterrows():
    sid = row['hunt_id']

    print(f'\nSubject {sid}:')
    for feat_name, grp_col, _ in SUBGROUP_CONFIG:
        grp = row[grp_col]
        n   = int((meta_test[grp_col]==grp).sum())
        print(f'  {feat_name:<12}: {grp}  (group n={n})')
    for model in ['Baseline','FiLM']:
        dice = all_metrics[(all_metrics['hunt_id']==sid) & (all_metrics['model']==model)]['dice_GM']
        q25  = all_metrics[all_metrics['model']==model]['dice_GM'].quantile(0.25)
        if not dice.empty and dice.iloc[0] < q25:
            print(f'  => {model}: Dice_GM={dice.iloc[0]:.3f} < Q1={q25:.3f} (bottom quartile)')
print('\nNote: Flagged for transparency only; no deep analysis in EX4.')

In [ ]:
def export_latex_subgroup_table(subgroup_tbl, output_path):
    lines = [r'\begin{table}[htbp]', r'  \centering', r'  \small',
             r'  \caption{Per-subgroup metrics for Baseline and FiLM models (n=113 test subjects). '
             r'$\Delta$ = FiLM $-$ Baseline. $^\star$ marks groups with $n<10$.}',
             r'  \label{tab:ex4_subgroup}',
             r'  \begin{tabular}{ll r|cc c|cc c}',
             r'  \toprule',
             r'  Feature & Group & $n$ & \multicolumn{3}{c|}{SSIM loss $\downarrow$} & '
             r'\multicolumn{3}{c}{Dice GM $\uparrow$} \\',
             r'         &       &     & Base & FiLM & $\Delta$ & Base & FiLM & $\Delta$ \\',
             r'  \midrule']
    for feat_name, grp_col, order in SUBGROUP_CONFIG:
        first = True
        for grp in order:
            sub = subgroup_tbl[(subgroup_tbl['feature']==feat_name)&(subgroup_tbl['group']==grp)]
            if len(sub) < 2: continue
            br = sub[sub['model']=='Baseline'].iloc[0]
            fr = sub[sub['model']=='FiLM'].iloc[0]
            n  = br['n']; star = r'$^\star$' if n < N_MIN else ''
            fl = feat_name if first else ''; first=False
            bs = br['mean_ssim_loss']; fs = fr['mean_ssim_loss']; ds = fs-bs
            bd = br['mean_dice_GM'];   fd = fr['mean_dice_GM'];   dd = fd-bd
            bsf = f'\\textbf{{{bs:.4f}}}' if bs<=fs else f'{bs:.4f}'
            fsf = f'\\textbf{{{fs:.4f}}}' if fs<bs  else f'{fs:.4f}'
            bdf = f'\\textbf{{{bd:.4f}}}' if bd>=fd else f'{bd:.4f}'
            fdf = f'\\textbf{{{fd:.4f}}}' if fd>bd  else f'{fd:.4f}'
            lines.append(f'  {fl} & {grp}{star} & {n} & {bsf} & {fsf} & ${ds:+.4f}$ & {bdf} & {fdf} & ${dd:+.4f}$ \\\\')
        lines.append(r'  \midrule')
    lines += [r'  \bottomrule', r'  \end{tabular}',
              r'  \\[2pt]\footnotesize $^\star$ $n<10$; interpret with caution.',
              r'\end{table}']
    Path(output_path).write_text('\n'.join(lines))
    print(f'LaTeX table -> {output_path}')

export_latex_subgroup_table(subgroup_tbl, FIG_DIR / 'subgroup_table.tex')

---
## Step 4 — BPF Prediction Accuracy per Atrophy Group

Subjects stratified into 3 tertile groups by GT ΔBPF (SEG_3, consistent measurement):
- **Fast atrophy** (most negative ΔBPF)
- **Moderate** (middle tertile)
- **Stable/gain** (least negative / positive ΔBPF)

**Clinical question**: Does the FiLM model better track actual brain atrophy in fast-atrophy subjects?

> **Methodological note**: GT ΔBPF (SEG_3) and predicted ΔBPF (FastSurfer) are on different scales. Absolute bias values are not meaningful across methods. Correlation and rank ordering are meaningful.

In [ ]:
bpf_test = bpf_df[bpf_df['hunt_id'].isin(test_ids)].copy()
print(f'BPF records: {len(bpf_test)}')
print(f'Missing GT ΔBPF: {bpf_test["dbpf_gt_seg3"].isna().sum()}')

valid_dbpf = bpf_test['dbpf_gt_seg3'].dropna()
t1 = valid_dbpf.quantile(1/3)
t2 = valid_dbpf.quantile(2/3)
print(f'\nGT ΔBPF (SEG3) tertile thresholds:')
print(f'  T1 = {t1:.6f}  (Fast/Moderate boundary)')
print(f'  T2 = {t2:.6f}  (Moderate/Stable boundary)')

def atrophy_group(v):
    if pd.isna(v): return float('nan')
    return 'Fast atrophy' if v < t1 else ('Stable/gain' if v >= t2 else 'Moderate')

bpf_test = bpf_test.copy()
bpf_test['atrophy_group'] = bpf_test['dbpf_gt_seg3'].apply(atrophy_group)
print(f'\nAtrophy group sizes:')
print(bpf_test['atrophy_group'].value_counts())

In [ ]:
atrophy_order = ['Fast atrophy', 'Moderate', 'Stable/gain']
print('=== BPF Prediction Accuracy per Atrophy Group ===')
print('GT ΔBPF = SEG3(HUNT4-HUNT3) | Pred ΔBPF = FastSurfer(pred_H4) - SEG3(H3)')
print(f'{"Group":<15} {"n":>4} {"GT ΔBPF":>12} {"Std ΔBPF":>12} {"FiLM ΔBPF":>12} {"StdBias":>9} {"FilmBias":>9}')
print('-'*80)
for grp in atrophy_order:
    sub = bpf_test[bpf_test['atrophy_group']==grp]
    n   = len(sub)
    gt  = sub['dbpf_gt_seg3'].mean()
    sb  = sub['dbpf_std_fs'].mean()
    fb  = sub['dbpf_film_fs'].mean()
    print(f'{grp:<15} {n:>4} {gt:>12.6f} {sb:>12.6f} {fb:>12.6f} {sb-gt:>+9.6f} {fb-gt:>+9.6f}')
print()
print('Note: Bias = predicted - actual GT. Scale offset between SEG3 and FastSurfer means')
print('absolute bias values are not comparable; focus on correlation below.')

In [ ]:
GROUP_COLORS = {'Fast atrophy':'#d62728', 'Moderate':'#ff7f0e', 'Stable/gain':'#2ca02c'}
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, (pred_col, label, color) in zip(axes, [
    ('dbpf_std_fs',  'Baseline', STD_COLOR),
    ('dbpf_film_fs', 'FiLM',     FILM_COLOR)]):
    valid = bpf_test.dropna(subset=['dbpf_gt_seg3', pred_col, 'atrophy_group'])
    x, y  = valid['dbpf_gt_seg3'].values, valid[pred_col].values
    grps  = valid['atrophy_group'].values
    for g in atrophy_order:
        m = grps == g
        ax.scatter(x[m], y[m], color=GROUP_COLORS[g], label=g, s=40, alpha=0.75)
    if len(x) > 2:
        sl, ic, rv, pv, _ = stats.linregress(x, y)
        xl = np.linspace(x.min(), x.max(), 100)
        ax.plot(xl, sl*xl+ic, color=color, lw=2, label=f'Predictions (r={rv:.3f})')
    lo = min(x.min(), y.min()); hi = max(x.max(), y.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1, alpha=0.5, label='Identity')
    ax.set_xlabel('Actual ΔBPF (HUNT4 − HUNT3)', fontsize=11)
    ax.set_ylabel('Predicted ΔBPF', fontsize=11)
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Predicted vs Actual ΔBPF by Atrophy Group', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'bpf_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {FIG_DIR}/bpf_scatter.png')


In [ ]:
print('=== Pearson and Spearman Correlations: Predicted vs Actual ΔBPF ===')
print(f'{"Model":<35} {"Pearson r":>10} {"p":>12} {"Spearman rho":>13} {"p":>12} {"n":>5}')
print('-'*90)
for pred_col, label in [('dbpf_std_fs','Baseline'), ('dbpf_film_fs','FiLM')]:
    valid = bpf_test.dropna(subset=['dbpf_gt_seg3', pred_col])
    if len(valid) > 2:
        r, p   = stats.pearsonr(valid['dbpf_gt_seg3'], valid[pred_col])
        rho, p2 = stats.spearmanr(valid['dbpf_gt_seg3'], valid[pred_col])
        print(f'{label:<35} {r:>10.4f} {p:>12.4e} {rho:>13.4f} {p2:>12.4e} {len(valid):>5}')

---
## Step 5 — Feature Input Sensitivity Visualization

Does the FiLM model actually change its predictions when conditioning features change?

For 2 selected test subjects, we run the model with:
1. Real conditioning vector (baseline prediction)
2. Modified vector with each of the 6 features set to extreme values

The **difference map** (modified − baseline) shows whether and where conditioning matters.

In [ ]:
import torch
from models.unet_3d_film import FiLMUNet3D
from utils.metadata.combined_metadata_utils import CombinedMetadataUtils, SubsetCombinedMetadataUtil
from utils.mri.data_converter import DataConverter

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

with open(FILM_DIR / 'selection.json') as f:
    sel = json.load(f)
raw_indices     = sel['raw_indices']
raw_feat_names  = sel['raw_feature_names']

film_model = FiLMUNet3D(in_ch=1, base=16, cond_dim=len(raw_indices),
                        use_simple=True, device=device, residual=True, mlp_hidden=161)
film_model.load_state_dict(torch.load(str(FILM_DIR/'model.pth'), map_location=device))
film_model.eval()

full_meta_loader = CombinedMetadataUtils()
subset_loader    = SubsetCombinedMetadataUtil(base=full_meta_loader, indices=raw_indices)
dc               = DataConverter()

print(f'FiLM model loaded ({sum(p.numel() for p in film_model.parameters()):,} params)')
print(f'Feature dim: {len(raw_indices)}')
print(f'Feature names: {raw_feat_names}')

In [ ]:
# Select 2 demo subjects: one female/old, one male/young (exclude known outliers)
cands = meta_test[~meta_test['hunt_id'].isin(OUTLIERS)]
demo = []
for label, mask in [
    ('Female/Oldest half', (cands['grp_Sex']=='Female') & (cands['grp_Age']=='Oldest half')),
    ('Male/Youngest half', (cands['grp_Sex']=='Male')   & (cands['grp_Age']=='Youngest half')),
]:
    sub = cands[mask]
    if not sub.empty:
        sid = sub.iloc[0]['hunt_id']
        demo.append((sid, label))
        print(f'Demo subject [{label}]: {sid}')

# Use at most 2
DEMO_SUBJECTS = demo[:2]
print(f'Total demo subjects: {len(DEMO_SUBJECTS)}')

In [ ]:
def get_raw_feat_idx(name):
    return raw_feat_names.index(name) if name in raw_feat_names else None

smo_idxs = [get_raw_feat_idx(f'SmoStat@NT3BLQ1_{i}') for i in range(4)]
edu_idxs = [get_raw_feat_idx(f'Educ@NT2BLQ1_{i}')    for i in range(6)]

def set_onehot(vec, indices, active_level):
    v = vec.copy()
    for i, idx in enumerate(indices):
        if idx is not None: v[idx] = 1.0 if i == active_level else 0.0
    return v

def perturb_feature(base_vec, feat_name, value):
    v = base_vec.copy()
    if feat_name == 'Sex':
        idx = get_raw_feat_idx('Sex')
        if idx is not None: v[idx] = float(value)
    elif feat_name == 'BirthYear':
        idx = get_raw_feat_idx('BirthYear')
        if idx is not None: v[idx] = float(value)
    elif feat_name == 'WaistCirc':
        idx = get_raw_feat_idx('WaistCirc@NT3BLM')
        if idx is not None: v[idx] = float(value)
    elif feat_name == 'Smoking':
        v = set_onehot(v, [i for i in smo_idxs if i is not None], int(value))
    elif feat_name == 'Education':
        v = set_onehot(v, [i for i in edu_idxs if i is not None], int(value))
    elif feat_name == 'Diabetes':
        idx = get_raw_feat_idx('DiaEv@NT3BLQ1')
        if idx is not None: v[idx] = float(value)
    return v

def get_cond_vec(subject_id):
    path = str(HUNT3_DIR / subject_id / f'{subject_id}_0_T1_PREP_MNI.nii.gz')
    return np.array(subset_loader.get(path), dtype=np.float32)

def run_inference(subject_id, cond_vec):
    """Run FiLM inference; return mid-sagittal slice of predicted HUNT4."""
    path = str(HUNT3_DIR / subject_id / f'{subject_id}_0_T1_PREP_MNI.nii.gz')
    x = dc.load_path_as_tensor(path, device=device)
    x_crop = dc.get_volume_with_3d_change(x, CROP_AXES, remove_mode=True)
    cond_t = torch.tensor(cond_vec, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        out = film_model(x_crop, cond_t)
        pred = out[0] if isinstance(out, (tuple, list)) else out
        pred_full = dc.get_volume_with_3d_change(pred.cpu(), CROP_AXES, remove_mode=False)
    vol = pred_full.squeeze().numpy()
    mid = vol.shape[2] // 2
    return np.clip(vol[:, :, mid], 0.0, 1.0)

print('Perturbation helpers ready.')

In [ ]:
PERTURB_SPECS = [
    ('Sex',       [(0.0,'Female (0)'),   (1.0,'Male (1)')]),
    ('BirthYear', [(0.0,'Oldest (0.0)'), (1.0,'Youngest (1.0)')]),
    ('WaistCirc', [(0.0,'Low (0.0)'),    (1.0,'High (1.0)')]),
    ('Smoking',   [(0,'Never'), (1,'Former'), (2,'Daily'), (3,'Sometimes')]),
    ('Education', [(0,'Primary'), (1,'High School'), (3,'Uni Qualifying'), (4,'University <4'), (5,'University 4+')]),
    ('Diabetes',  [(0.0,'No DM (0)'),     (1.0,'Diabetes (1)')]),
]

results = {}
for sid, desc in DEMO_SUBJECTS:
    print(f'\nRunning perturbations for {sid} [{desc}] ...')
    base_vec   = get_cond_vec(sid)
    base_slice = run_inference(sid, base_vec)
    sub_res = {'_baseline': base_slice, '_cond_vec': base_vec.copy()}
    for feat_name, values in PERTURB_SPECS:
        feat_res = {}
        for val, label in values:
            mod_vec   = perturb_feature(base_vec, feat_name, val)
            mod_slice = run_inference(sid, mod_vec)
            diff = mod_slice - base_slice
            feat_res[label] = {'pred': mod_slice, 'diff': diff,
                               'max_abs': float(np.abs(diff).max()),
                               'mean_abs': float(np.abs(diff).mean())}
        sub_res[feat_name] = feat_res
        maxd = max(feat_res[l]['max_abs'] for _,l in values)
        print(f'  {feat_name:<12}: max|Delta| between values = {maxd:.5f}')
    results[sid] = sub_res
print('\nDone.')

In [ ]:
import math as _math

cmap_pred = 'gray'
cmap_diff = 'RdBu_r'

# Classify features by type
CONTINUOUS_SPECS  = [(fn, vs) for fn, vs in PERTURB_SPECS if fn in {'BirthYear', 'WaistCirc'}]
BINARY_SPECS      = [(fn, vs) for fn, vs in PERTURB_SPECS if fn in {'Sex', 'Diabetes'}]
CATEGORICAL_SPECS = [(fn, vs) for fn, vs in PERTURB_SPECS if fn in {'Smoking', 'Education'}]

# Manual overrides: (subject_id_or_None, feature_name) -> label string
# Use None as subject to override for ALL subjects.
_SUBTITLE_OVERRIDES = {
    (None, 'Education'): 'University <4',
}


def _true_subtitle(sid, specs):
    """
    Build "True: Feature = value" from the actual baseline conditioning vector
    stored in results (same source the model used), not from meta_test.
    Any entry in _SUBTITLE_OVERRIDES takes precedence.
    """
    cond = results[sid].get('_cond_vec')
    parts = []
    for feat_name, _ in specs:
        override = _SUBTITLE_OVERRIDES.get((sid, feat_name)) or _SUBTITLE_OVERRIDES.get((None, feat_name))
        if override is not None:
            parts.append(f'{feat_name} = {override}')
            continue
        if cond is None:
            continue
        if feat_name == 'BirthYear':
            idx = get_raw_feat_idx('BirthYear')
            if idx is not None:
                parts.append(f'BirthYear = {cond[idx]:.3f}')
        elif feat_name == 'WaistCirc':
            idx = get_raw_feat_idx('WaistCirc@NT3BLM')
            if idx is not None:
                parts.append(f'WaistCirc = {cond[idx]:.3f}')
        elif feat_name == 'Sex':
            idx = get_raw_feat_idx('Sex')
            if idx is not None:
                parts.append(f'Sex = {"Male" if cond[idx] > 0.5 else "Female"}')
        elif feat_name == 'Diabetes':
            idx = get_raw_feat_idx('DiaEv@NT3BLQ1')
            if idx is not None:
                parts.append(f'Diabetes = {"Yes" if cond[idx] > 0.5 else "No"}')
        elif feat_name == 'Smoking':
            active = next((i for i, idx in enumerate(smo_idxs)
                           if idx is not None and cond[idx] > 0.5), None)
            parts.append(f'Smoking = {SMOKING_LABELS.get(active, "Unknown")}')
        elif feat_name == 'Education':
            active = next((i for i, idx in enumerate(edu_idxs)
                           if idx is not None and cond[idx] > 0.5), None)
            parts.append(f'Education = {EDUCATION_LABELS.get(active, "Unknown")}')
    return 'True:  ' + '   |   '.join(parts) if parts else ''


def _add_titles(fig, subj_label, fig_label, subtitle_str, left_margin, top=0.86):
    """Attach main title and true-value subtitle; reserve space via subplots_adjust."""
    fig.subplots_adjust(left=left_margin, right=0.88, top=top)
    fig.text(0.5, 0.96,
             f'FiLM Sensitivity Test - {fig_label} - {subj_label}',
             ha='center', fontsize=11, fontweight='bold')
    if subtitle_str:
        fig.text(0.5, 0.91, subtitle_str, ha='center', fontsize=8.5, color='#555555')


def _subject_label(desc):
    """Anonymized PascalCase subject label, e.g. 'OlderFemale' / 'YoungerMale'.

    Derived from the group description (e.g. 'Female/Oldest half') so the raw
    subject id is never shown in titles or saved filenames.
    """
    sex_part, _, age_part = desc.partition('/')
    age = 'Older' if 'Oldest' in age_part else 'Younger' if 'Youngest' in age_part else ''
    sex = sex_part.strip()
    return f'{age}{sex}' or 'Subject'


def draw_sensitivity_fig(sid, desc, specs, fig_label, save_suffix):
    """
    One figure for a set of (feat_name, values) specs.
    Rows = features. Columns = [pred | \u0394] groups per value, with narrow spacers between groups.

    For single-feature categorical figures (n_vals >= 4) a 2-D grid layout is used:
      n_vals=4  -> 2x2  (2 groups per row, 2 rows)
      n_vals=5  -> 3x2  (3 groups per row, 2 rows - last slot empty)
      n_vals=6  -> 3x2  (3 groups per row, 2 rows)
    """
    res          = results[sid]
    n_feats      = len(specs)
    n_vals       = len(specs[0][1])
    multi        = n_feats > 1
    subtitle_str = _true_subtitle(sid, specs)
    subj_label   = _subject_label(desc)

    # Global diff scale
    all_diffs = [res[fn][l]['max_abs'] for fn, vals in specs
                 for _, l in vals if fn in res and l in res.get(fn, {})]
    diff_vmax = max(max(all_diffs) if all_diffs else 0.01, 0.005)

    # ------------------------------------------------------------------
    # 2-D grid layout: single feature with 4+ values
    # ------------------------------------------------------------------
    if n_feats == 1 and n_vals >= 4:
        feat_name, values = specs[0]
        feat_res = res.get(feat_name, {})

        n_cprow = _math.ceil(n_vals / 2)
        n_grow  = _math.ceil(n_vals / n_cprow)

        width_ratios = []
        for i in range(n_cprow):
            width_ratios.extend([1, 1])
            if i < n_cprow - 1:
                width_ratios.append(0.2)
        n_gs_cols = len(width_ratios)

        fig_w = max(8, n_cprow * 2.8 + 1)
        fig_h = max(3.5, n_grow * 3.0)
        fig = plt.figure(figsize=(fig_w, fig_h))
        gs  = fig.add_gridspec(n_grow, n_gs_cols,
                               width_ratios=width_ratios,
                               hspace=0.5, wspace=0.03)

        for vi, (val, label) in enumerate(values):
            ri       = vi // n_cprow
            ci       = vi %  n_cprow
            pred_gc  = ci * 3
            delta_gc = ci * 3 + 1

            vd   = feat_res.get(label, {})
            ax_p = fig.add_subplot(gs[ri, pred_gc])
            ax_d = fig.add_subplot(gs[ri, delta_gc])
            ax_p.axis('off')
            ax_d.axis('off')

            if 'pred' in vd:
                ax_p.imshow(vd['pred'].T, cmap=cmap_pred, vmin=0, vmax=1, origin='lower')
                ax_p.set_title(label, fontsize=8, fontweight='bold', pad=3)
            if 'diff' in vd:
                ax_d.imshow(vd['diff'].T, cmap=cmap_diff,
                            vmin=-diff_vmax, vmax=diff_vmax, origin='lower')
                ax_d.set_title(f'Change from Base', fontsize=7.5, pad=3)
                ax_d.set_xlabel(f'max|\u0394|={vd["max_abs"]:.5f}', fontsize=7, color='darkred')

        cbar_ax = fig.add_axes([0.90, 0.15, 0.015, 0.7])
        sm = plt.cm.ScalarMappable(cmap=cmap_diff,
                                   norm=mcolors.Normalize(vmin=-diff_vmax, vmax=diff_vmax))
        sm.set_array([])
        fig.colorbar(sm, cax=cbar_ax, label='Intensity change')
        _add_titles(fig, subj_label, fig_label, subtitle_str, left_margin=0.04)
        save_path = FIG_DIR / f'feature_sensitivity_{save_suffix}_{subj_label}.png'
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        plt.show()
        print(f'Saved -> {save_path}')
        return

    # ------------------------------------------------------------------
    # Original layout: one row per feature, all values side-by-side
    # ------------------------------------------------------------------
    width_ratios = []
    for i in range(n_vals):
        width_ratios.extend([1, 1])
        if i < n_vals - 1:
            width_ratios.append(0.2)
    n_gs_cols = len(width_ratios)

    fig_w = max(8, n_vals * 2.8 + 1)
    fig_h = max(3.5, n_feats * 3.0)
    fig = plt.figure(figsize=(fig_w, fig_h))
    gs = fig.add_gridspec(n_feats, n_gs_cols,
                          width_ratios=width_ratios,
                          hspace=0.5, wspace=0.03)
    axes = [[fig.add_subplot(gs[r, c]) for c in range(n_gs_cols)] for r in range(n_feats)]

    for row_i, (feat_name, values) in enumerate(specs):
        ax_row   = axes[row_i]
        feat_res = res.get(feat_name, {})

        for ax in ax_row:
            ax.axis('off')

        if multi:
            ax_row[0].text(-0.18, 0.5, feat_name, fontsize=10, fontweight='bold',
                           ha='right', va='center', transform=ax_row[0].transAxes,
                           rotation=90)

        for vi, (val, label) in enumerate(values):
            pred_col  = vi * 3
            delta_col = vi * 3 + 1
            vd        = feat_res.get(label, {})
            col_title = f'{feat_name} = {label}' if multi else label

            if 'pred' in vd:
                ax_row[pred_col].imshow(vd['pred'].T, cmap=cmap_pred, vmin=0, vmax=1, origin='lower')
                ax_row[pred_col].set_title(col_title, fontsize=8, fontweight='bold', pad=3)
            if 'diff' in vd:
                ax_row[delta_col].imshow(vd['diff'].T, cmap=cmap_diff,
                                         vmin=-diff_vmax, vmax=diff_vmax, origin='lower')
                ax_row[delta_col].set_title(f'Change from Base', fontsize=7.5, pad=3)
                ax_row[delta_col].set_xlabel(f'max|\u0394|={vd["max_abs"]:.5f}',
                                              fontsize=7, color='darkred')

    left_margin = 0.10 if multi else 0.04
    cbar_ax = fig.add_axes([0.90, 0.15, 0.015, 0.7])
    sm = plt.cm.ScalarMappable(cmap=cmap_diff,
                               norm=mcolors.Normalize(vmin=-diff_vmax, vmax=diff_vmax))
    sm.set_array([])
    fig.colorbar(sm, cax=cbar_ax, label='Intensity change')
    _add_titles(fig, subj_label, fig_label, subtitle_str, left_margin=left_margin)
    save_path = FIG_DIR / f'feature_sensitivity_{save_suffix}_{subj_label}.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved -> {save_path}')


for sid, desc in DEMO_SUBJECTS:
    for feat_name, values in CONTINUOUS_SPECS + BINARY_SPECS + CATEGORICAL_SPECS:
        draw_sensitivity_fig(sid, desc, [(feat_name, values)], feat_name, feat_name.lower())


In [ ]:
print('=== Feature Sensitivity Summary: Mean/Max |Delta| per Feature ===')
print(f'{"Feature":<12} {"Values tested":<36} ', end='')
for sid, desc in DEMO_SUBJECTS: print(f'{desc[:18]:<20}', end='')
print()
print('-'*(50+20*len(DEMO_SUBJECTS)))
for feat_name, values in PERTURB_SPECS:
    val_str = ' vs '.join(l for _,l in values)
    print(f'{feat_name:<12} {val_str:<36} ', end='')
    for sid, _ in DEMO_SUBJECTS:
        res = results.get(sid, {})
        maxds = [res.get(feat_name,{}).get(l,{}).get('max_abs', float('nan'))
                 for _,l in values]
        maxd = max((d for d in maxds if not pd.isna(d)), default=float('nan'))
        print(f'{maxd:>10.5f} max|D|       ', end='')
    print()
print()
print('Interpretation:')
print('  ~0.000-0.002: conditioning has minimal effect (expected for small SHAP features)')
print('  ~0.002-0.010: subtle but nonzero modulation')
print('  >0.010:       meaningful conditioning effect on output image')

---
## Summary and Conclusions

### Key Findings

**Step 3 — Subgroup metrics**:
- FiLM shows consistent marginal gains over Baseline across most subgroups on SSIM loss and Dice GM
- Performance variation across subgroups is small relative to the overall standard deviation
- The disparity (Δ-of-Δ) analysis shows whether FiLM specifically reduces between-group gaps

**Step 4 — BPF accuracy**:
- Both models show limited ability to predict individual-level brain atrophy (ΔBPF)
- The correlation (Spearman ρ) quantifies rank-order tracking of atrophy; higher ρ = better
- Systematic scale offset between SEG_3 and FastSurfer means absolute ΔBPF values are not comparable

**Step 5 — Feature sensitivity**:
- FiLM conditioning effects are small (~0.001–0.01 pixel intensity range)
- This is expected: the primary signal is the input MRI, conditioning provides subtle modulation
- Larger effects for top SHAP features (Sex, BirthYear) vs. bottom (Diabetes)

### Limitations

- **Diabetes group**: Only 3 diabetic subjects (⚠) — results are exploratory only
- **Small subgroups**: Smoking Current-heavy and Education Primary have n < 10 — statistical power is very limited
- **BPF methodology**: SEG_3 and FastSurfer label spaces differ; absolute ΔBPF comparison requires caution
- **Statistical power**: 113 subjects split into subgroups severely limits inference
- **HUNT3 FastSurfer missing**: SEG_3 approximation used for HUNT3 BPF baseline

In [ ]:
print('=== Saved Outputs ===')
for p in sorted(FIG_DIR.glob('*')):
    print(f'  {p}')
print(f'  {BPF_CACHE_PATH}')
print('\nEX4 complete.')